In [10]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import pdist, squareform, cdist
import os
from tqdm import tqdm

# ================= 配置 =================
# 读取之前生成的 h5ad (包含 Embedding 和 spatial)
H5AD_PATH = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/LungCancer_10x_result.h5ad"
RESULT_DIR = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/"

def calculate_spatial_gradient(adata, n_neighbors=15):
    """
    流程图复现：
    1. 输入: Embeddings (adata.X)
    2. 找邻居: 基于物理空间 (adata.obsm['spatial'])
    3. 对比: 计算中心点与邻居在特征空间上的距离 (Cosine Distance)
    4. 输出: 梯度得分 (Gradient Score)
    """
    print("[-] 正在执行空间梯度分析...")
    
    # [Box 1] 获取输入
    coords = adata.obsm['spatial'] # 物理坐标
    features = adata.X             # H Embeddings (深度特征)
    
    # [Box 2] 找邻居 (Spot物理距离)
    # 寻找每个点在物理空间上最近的 n 个邻居
    print("    -> Step 1: 寻找物理邻居 (KNN)...")
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(coords)
    _, indices = nbrs.kneighbors(coords)
    
    # [Box 3] 对比 (特征向量)
    # 计算每个点，与其周围邻居在"特征空间"上的平均差异
    print("    -> Step 2: 计算特征差异 (Gradient)...")
    gradient_scores = []
    
    # 遍历每一个点 (可以使用矩阵运算加速，但为了逻辑清晰这里用循环演示)
    for i in tqdm(range(len(coords)), desc="Computing"):
        # 当前点的特征
        current_feat = features[i].reshape(1, -1)
        
        # 邻居点的特征
        neighbor_feats = features[indices[i]]
        
        # 计算当前点和它所有邻居的余弦距离 (Cosine Distance)
        # 距离越小(0)越相似，距离越大(1)差异越大
        dists = cdist(current_feat, neighbor_feats, metric='cosine')
        
        # [Box 4] 边界得分
        # 取平均距离作为该位置的"混乱度"或"梯度值"
        avg_dist = np.mean(dists)
        gradient_scores.append(avg_dist)
        
    return np.array(gradient_scores)

def visualize_gradient(h5ad_path):
    # 1. 加载数据
    if not os.path.exists(h5ad_path):
        print(f"❌ 文件不存在: {h5ad_path}")
        return
    adata = sc.read_h5ad(h5ad_path)
    
    # 2. 计算梯度
    scores = calculate_spatial_gradient(adata, n_neighbors=20)
    adata.obs['gradient_score'] = scores
    
    # ==========================================
    # 【修改点】自动计算更大的 Spot Size
    # ==========================================
    coords = adata.obsm['spatial']
    span_x = coords[:, 0].max() - coords[:, 0].min()
    span_y = coords[:, 1].max() - coords[:, 1].min()
    max_span = max(span_x, span_y)
    
    # 系数设为 0.025 (2.5%)，让点更大，形成连续的视觉效果
    # 如果你觉得还不够大，可以改成 0.03 或 0.04
    auto_spot_size = max_span * 0.025 
    
    # 设个下限，防止太小
    auto_spot_size = max(auto_spot_size, 100)
    
    print(f"[-] 坐标跨度: {max_span:.1f}")
    print(f"[-] 自动调整 Spot Size 为: {auto_spot_size:.1f} (为了更好的梯度视觉效果)")
    # ==========================================

    # 3. 绘图
    print("[-] 正在绘图...")
    plt.figure(figsize=(8, 8))
    
    sc.pl.spatial(
        adata,
        color='gradient_score',
        cmap='plasma',  # 推荐用 plasma 或 magma，亮色代表高梯度
        title='Spatial Gradient Analysis',
        spot_size=auto_spot_size,  # <--- 使用这一步计算出的大点
        show=False,
        # 截断前 5% 和后 5% 的极端值，让颜色对比度更强
        vmin=np.percentile(scores, 5), 
        vmax=np.percentile(scores, 95)
    )
    
    save_path = os.path.join(RESULT_DIR, "spatial_gradient_continuous.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ 梯度分析图已保存: {save_path}")

if __name__ == "__main__":
    visualize_gradient(H5AD_PATH)

[-] 正在执行空间梯度分析...
    -> Step 1: 寻找物理邻居 (KNN)...
    -> Step 2: 计算特征差异 (Gradient)...


Computing: 100%|██████████| 4415/4415 [00:00<00:00, 60691.19it/s]

[-] 坐标跨度: 24514.0
[-] 自动调整 Spot Size 为: 612.9 (为了更好的梯度视觉效果)
[-] 正在绘图...


✅ 梯度分析图已保存: /data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/spatial_gradient_continuous.png


<Figure size 800x800 with 0 Axes>

In [12]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import os

# ================= 配置路径 =================
# 你的输入文件路径
H5AD_PATH = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/ProstateAcinarCancer_10x_result.h5ad"
# 结果保存文件夹
RESULT_DIR = "/data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/"

# ================= 辅助函数：标签平滑 (已修复) =================
def refine_labels(adata, domain_key='spatial_domain', n_neighbors=20):
    """
    使用 KNN 投票机制平滑空间标签。
    【修复】: 使用 np.unique 替代 scipy.stats.mode，解决字符串标签报错问题。
    """
    print(f"[-] 正在对 {domain_key} 进行平滑处理 (n_neighbors={n_neighbors})...")
    
    # 获取坐标
    coords = adata.obsm['spatial']
    # 获取原始标签 (确保是 numpy 数组)
    labels = adata.obs[domain_key].values.copy()
    
    # 构建邻居图
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(coords)
    _, indices = nbrs.kneighbors(coords)
    
    new_labels = labels.copy()
    
    # 遍历每个点，让它服从"大多数邻居"的类别
    for i in range(len(coords)):
        neighbor_labels = labels[indices[i]]
        
        # --- 【关键修改】使用 numpy 计算众数 ---
        # unique_vals: 出现的唯一标签
        # counts: 每个标签出现的次数
        unique_vals, counts = np.unique(neighbor_labels, return_counts=True)
        
        # 找到出现次数最多的那个标签的索引
        most_common_index = np.argmax(counts)
        most_common = unique_vals[most_common_index]
        # ------------------------------------
        
        new_labels[i] = most_common
        
    return new_labels

# ================= 主处理函数 =================
def process_prostate_analysis(h5ad_path):
    if not os.path.exists(h5ad_path):
        print(f"❌ 找不到文件: {h5ad_path}")
        return

    # 1. 加载数据
    print("[-] 正在加载数据...")
    adata = sc.read_h5ad(h5ad_path)
    
    # 检查是否包含聚类结果
    if 'spatial_domain' not in adata.obs:
        print("⚠️ 未检测到 'spatial_domain'，正在重新运行 Leiden 聚类...")
        sc.pp.neighbors(adata, use_rep='X', n_neighbors=15, metric='cosine')
        sc.tl.leiden(adata, resolution=0.5, key_added='spatial_domain')

    # 2. 自动计算 Spot Size
    coords = adata.obsm['spatial']
    span = max(coords[:, 0].max() - coords[:, 0].min(), 
               coords[:, 1].max() - coords[:, 1].min())
    auto_spot_size = max(span * 0.018, 50)
    print(f"[-] 自动计算点大小: {auto_spot_size:.1f}")

    # 3. 执行标签平滑 (使用修复后的函数)
    refined_labels = refine_labels(adata, domain_key='spatial_domain', n_neighbors=25)
    adata.obs['spatial_domain_refined'] = refined_labels

    # 4. 绘图
    print("[-] 正在绘图...")
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    
    # 左图：原始聚类
    sc.pl.spatial(
        adata, color='spatial_domain', title="Original Clustering (Raw)",
        spot_size=auto_spot_size, palette='tab20', ax=axs[0], show=False
    )
    
    # 右图：平滑后 (Refined)
    sc.pl.spatial(
        adata, color='spatial_domain_refined', title="Refined Spatial Domains (Smoothed)",
        spot_size=auto_spot_size, palette='tab20', ax=axs[1], show=False
    )
    
    plt.tight_layout()
    
    # 保存结果
    save_path = os.path.join(RESULT_DIR, "Prostate_Refined_Visualization.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # 5. 保存更新后的 h5ad
    adata.write(h5ad_path)
    print(f"✅ 图片已保存: {save_path}")
    print(f"✅ 数据已更新: {h5ad_path}")

if __name__ == "__main__":
    process_prostate_analysis(H5AD_PATH)

[-] 正在加载数据...
[-] 自动计算点大小: 350.8
[-] 正在对 spatial_domain 进行平滑处理 (n_neighbors=25)...
[-] 正在绘图...
✅ 图片已保存: /data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/Prostate_Refined_Visualization.png
✅ 数据已更新: /data/home/wangzz_group/zhaipengyuan/BEPH-main/Visualization/spatial_domains_result/ProstateAcinarCancer_10x_result.h5ad
